# 조선일보 기사 데이터 전처리

**목표** — 원본 `기사 본문 전체` 컬럼에서 실제 기사 본문 / 작성자(기자)를 분리해내고, 이상치·결측치를 정제한다.

**핵심 이슈** — `기사 본문 전체` 컬럼은 기사 본문이 아니라 **렌더링된 웹페이지 전체 텍스트**다 (평균 15,294자, 이 중 약 91%가 보일러플레이트).

```
[상단 네비 메뉴 ×4회 반복] -> [섹션] -> [제목] -> [기자명] -> 입력 2025.01.01. 09:04
-> 업데이트 ... -> [댓글수] -> ★실제 본문★ -> [#해시태그/관련기사] -> [기자 프로필]
-> [구독수/100자평] -> [Taboola 광고] -> [많이 본 뉴스] -> [푸터 사이트맵 ×3회 반복]
```

**파싱 시 함정 2가지** (아래 코드에 반영됨)

1. 헤더 블록이 페이지에 **두 번 반복**된다. `입력`을 마지막 것 기준으로 자르면 바이라인이 head 중간에 묻혀 작성자 추출률이 32.8%로 떨어진다.
   -> **바이라인은 첫 번째 `입력` 앞**, **본문 시작은 마지막 타임스탬프 뒤**에서 잡아야 97.7%가 나온다.
2. 본문 뒤에 **기자 프로필**이 붙어 본문 끝을 오염시킨다.
   -> 추출한 작성자 이름을 **종료 앵커로 재활용**해서 차단한다.

**주의** — 페이지 헤더의 `2026년 4월 20일`은 **크롤링 시점**이지 기사 날짜가 아니다.

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)

RAW_PATH = Path("D:/멋사_부트캠프/클라비_기업프로젝트/data/AI_기반_뉴스_사실검증_시스템_프로젝트_데이터.csv")
OUT_PATH = Path("D:/멋사_부트캠프/클라비_기업프로젝트/data/processed_articles.csv")

MIN_BODY_LEN = 200  # 이 길이 미만은 본문 소실로 간주 (아래 분포 보고 조정 가능)

## 1. 원본 적재 및 구조 확인

In [2]:
raw = pd.read_csv(RAW_PATH)

print("shape:", raw.shape)
print("columns:", list(raw.columns))
print()
print("--- 결측치 ---")
print(raw.isnull().sum())
print()
print("--- 중복 ---")
print("URL 중복    :", raw["URL"].duplicated().sum())
print("제목 중복   :", raw["기사제목"].duplicated().sum())
print()
print("--- 레이블 분포 (True=키워드 검색, False=전체) ---")
print(raw["검색 구분 레이블"].value_counts())
print()
print("--- 작성일 범위 ---")
print(raw["작성일"].min(), "~", raw["작성일"].max(), " (명세는 12/31까지 -> 12/26~31 누락)")
print()
print("--- 원본 '기사 본문 전체' 길이: 기사가 아니라 페이지 전체 텍스트 ---")
print(raw["기사 본문 전체"].fillna("").str.len().describe().round(0))

shape: (2706, 5)
columns: ['기사제목', '작성일', 'URL', '기사 본문 전체', '검색 구분 레이블']

--- 결측치 ---
기사제목         0
작성일          0
URL          0
기사 본문 전체     1
검색 구분 레이블    0
dtype: int64

--- 중복 ---
URL 중복    : 10
제목 중복   : 12

--- 레이블 분포 (True=키워드 검색, False=전체) ---
검색 구분 레이블
True     2507
False     199
Name: count, dtype: int64

--- 작성일 범위 ---
2025-01-01 ~ 2025-12-25  (명세는 12/31까지 -> 12/26~31 누락)

--- 원본 '기사 본문 전체' 길이: 기사가 아니라 페이지 전체 텍스트 ---
count     2706.0
mean     15294.0
std       8055.0
min          0.0
25%      14065.0
50%      18161.0
75%      20750.0
max      33666.0
Name: 기사 본문 전체, dtype: float64


## 2. 파싱 규칙 정의

| 앵커 | 역할 |
|---|---|
| `TS_FIRST` | **첫** 타임스탬프. 이 앞부분(`head`)에서 바이라인을 뽑는다 |
| `TS_FULL` | `입력`+`업데이트`+`댓글수`까지 통째로. **마지막** 매치의 끝 = 본문 시작 |
| `BYLINE` | `head` 끝에 붙은 `홍길동 기자`, `김철수 특파원`, `박영희 기자(조선비즈)` |
| `END_MARK` | 본문 종료 지점. 해시태그·관련기사·프로필·광고·푸터의 시작 |

In [3]:
TS_FIRST = re.compile(r"입력\s*\d{4}\.\d{2}\.\d{2}\.\s*\d{1,2}:\d{2}")

TS_FULL = re.compile(
    r"입력\s*\d{4}\.\d{2}\.\d{2}\.\s*\d{1,2}:\d{2}"
    r"(?:\s*업데이트\s*\d{4}\.\d{2}\.\d{2}\.\s*\d{1,2}:\d{2})?"
    r"\s*(?:\d+\s*)?"  # 댓글 수
)

BYLINE = re.compile(r"((?:[가-힣]{2,4}\s*(?:기자|특파원)\s*(?:\([^)]{1,12}\))?\s*)+)$")

# 본문 말미의 검색용 해시태그 블록(#전기차, #북한 #러시아 ...)은 본문에서 제외한다.
# 단, '#' 하나만 앵커로 쓰면 아래 3건을 해시태그로 오인해 본문을 날린다.
#   [365]  "# 2024년 12월 현대차는..."   <- 리드 기호 (# 뒤 공백)   => 본문 전체 삭제됨
#   [561]  "#. 3년 차 사회 초년생..."    <- 리드 기호 (# 뒤 마침표) => 본문 전체 삭제됨
#   [1607] "'영웅문 S#'에서 오류가..."   <- 제품명 속 #            => 3760자가 240자로 잘림
# 따라서 '# 뒤에 공백 없이 글자가 바로 붙는' 형태까지 요구한다.
HASHTAG = r"#[가-힣A-Za-z0-9]"

END_MARK = (
    HASHTAG + r"|100자평|도움말\s*삭제기준|By\s*Taboola|많이\s*본\s*뉴스"
    r"|AI\s*추천|Copyright\s*조선일보|구독수\s*\d|더보기"
)


def parse_article(text: str) -> pd.Series:
    """페이지 전체 텍스트에서 (작성자, 본문, 포맷)을 분리한다."""
    first = TS_FIRST.search(text)
    if not first:
        # D형: 타임스탬프 앵커 자체가 없음 -> 유료화 기사의 프리뷰 스니펫
        return pd.Series([None, text.strip(), "D_프리뷰"])

    head = text[: first.start()].strip()                    # 바이라인은 '첫' 입력 앞
    body = text[list(TS_FULL.finditer(text))[-1].end() :]   # 본문은 '마지막' 타임스탬프 뒤

    m = BYLINE.search(head)
    if m:
        author = re.sub(r"\s+", " ", m.group(1)).strip()
    else:
        # 외부 필진(교수/소장 등)이거나 무기명
        author = "조선일보" if head.endswith("조선일보") else None

    marks = END_MARK
    if author and "기자" in author:
        # 본문 뒤에 재등장하는 '기자명 + 프로필' 블록을 종료 앵커로 추가
        marks += rf"|{re.escape(author.split()[0])}\s*기자"

    # re.search 는 leftmost match -> 여러 마커 중 가장 먼저 나오는 지점에서 자른다
    end = re.search(marks, body)
    fmt = "B_전체페이지" if end else "C_푸터없음"
    if end:
        body = body[: end.start()]

    return pd.Series([author, body.strip(), fmt])

## 3. 컬럼 추출

- `작성자` / `본문` / `포맷` — 페이지 텍스트 파싱
- `섹션` — 명세에는 있으나 데이터엔 없는 컬럼. URL 경로에서 100% 복원
- `제목` — 원본 `기사제목`이 이미 깨끗하므로 그대로 사용 (본문에서 재추출 불필요)

In [4]:
df = raw.copy()
df["기사 본문 전체"] = df["기사 본문 전체"].fillna("")

df[["작성자", "본문", "포맷"]] = df["기사 본문 전체"].apply(parse_article)

df["제목"] = df["기사제목"].str.strip()
df["섹션"] = df["URL"].str.extract(r"chosun\.com/([^/]+)/")
df["작성일"] = pd.to_datetime(df["작성일"])

# 본문 공백 정규화 (개행/연속 공백 -> 단일 공백)
df["본문"] = df["본문"].str.replace(r"\s+", " ", regex=True).str.strip()
df["본문길이"] = df["본문"].str.len()

print("--- 포맷 분포 ---")
print(df["포맷"].value_counts())
print()
print(f"작성자 추출률: {df['작성자'].notna().sum()} / {len(df)} "
      f"({df['작성자'].notna().mean() * 100:.1f}%) | 고유 기자 {df['작성자'].nunique()}명")
print()
print("--- 추출 본문 길이 (원본 평균 15,294자 -> ?) ---")
print(df["본문길이"].describe().round(0))

--- 포맷 분포 ---
포맷
B_전체페이지    2193
C_푸터없음      460
D_프리뷰        53
Name: count, dtype: int64

작성자 추출률: 2643 / 2706 (97.7%) | 고유 기자 322명

--- 추출 본문 길이 (원본 평균 15,294자 -> ?) ---
count    2706.0
mean     1395.0
std      1131.0
min         0.0
25%       695.0
50%      1004.0
75%      1806.0
max      7113.0
Name: 본문길이, dtype: float64


## 4. 이상치 / 결측치 정제

| 항목 | 처리 |
|---|---|
| 본문 빈 문자열 (크롤링 실패) | 제거 |
| `D_프리뷰` (유료화 스니펫, 본문 소실) | 제거 |
| 본문 `MIN_BODY_LEN` 미만 | 제거 |
| URL 중복 | 첫 행만 유지 |
| 제목+본문 완전 중복 | 첫 행만 유지 |

In [5]:
before = len(df)
log = []


def drop(mask: pd.Series, reason: str) -> None:
    """mask=True인 행을 제거하고 사유를 기록한다."""
    global df
    n = int(mask.sum())
    if n:
        df = df[~mask].copy()
    log.append((reason, n))


drop(df["본문"].eq(""), "본문 빈 문자열(크롤링 실패)")
drop(df["포맷"].eq("D_프리뷰"), "D_프리뷰(유료화 스니펫, 본문 소실)")
drop(df["본문길이"] < MIN_BODY_LEN, f"본문 {MIN_BODY_LEN}자 미만(저품질)")
drop(df["URL"].duplicated(keep="first"), "URL 중복")
drop(df.duplicated(subset=["제목", "본문"], keep="first"), "제목+본문 중복")

print("--- 제거 내역 ---")
for reason, n in log:
    print(f"  {reason:35s} : {n:4d}건")
print(f"\n{before}행 -> {len(df)}행  (총 {before - len(df)}건 제거, 잔존율 {len(df) / before * 100:.1f}%)")

--- 제거 내역 ---
  본문 빈 문자열(크롤링 실패)                    :    3건
  D_프리뷰(유료화 스니펫, 본문 소실)               :   52건
  본문 200자 미만(저품질)                     :   29건
  URL 중복                              :   10건
  제목+본문 중복                            :    0건

2706행 -> 2612행  (총 94건 제거, 잔존율 96.5%)


## 5. 검증

In [6]:
print("--- 결측치 ---")
print(df[["제목", "작성일", "섹션", "URL", "본문", "작성자", "검색 구분 레이블"]].isnull().sum())
print("  * 작성자 결측 = 외부 필진(교수/소장 등) 또는 무기명 -> 원본에 기자명이 없는 케이스")
print()
print("--- 보일러플레이트 제거 효과 ---")
print(f"  원본 평균 {raw['기사 본문 전체'].fillna('').str.len().mean():>8,.0f}자")
print(f"  본문 평균 {df['본문길이'].mean():>8,.0f}자")
print(f"  -> {(1 - df['본문길이'].mean() / raw['기사 본문 전체'].fillna('').str.len().mean()) * 100:.1f}% 가 보일러플레이트였음")
print()
print("--- 잔존 보일러플레이트 점검 (0이어야 정상) ---")
for kw in ["신문구독", "Copyright", "100자평", "Taboola", "많이 본 뉴스", "업데이트 2025"]:
    print(f"  '{kw}' 포함 본문: {df['본문'].str.contains(re.escape(kw)).sum()}건")
print()
print("--- 레이블 분포 (불균형 주의) ---")
vc = df["검색 구분 레이블"].value_counts()
print(vc)
print(f"  비율 {vc.max() / vc.min():.1f} : 1  -> 학습 시 class weight / 리샘플링 고려")
print()
print("--- 섹션 분포 ---")
print(df["섹션"].value_counts().head(8))
print()
print("--- 상위 기자 ---")
print(df["작성자"].value_counts().head(8))

--- 결측치 ---
제목            0
작성일           0
섹션            0
URL           0
본문            0
작성자          10
검색 구분 레이블     0
dtype: int64
  * 작성자 결측 = 외부 필진(교수/소장 등) 또는 무기명 -> 원본에 기자명이 없는 케이스

--- 보일러플레이트 제거 효과 ---
  원본 평균   15,294자
  본문 평균    1,431자
  -> 90.6% 가 보일러플레이트였음

--- 잔존 보일러플레이트 점검 (0이어야 정상) ---
  '신문구독' 포함 본문: 0건
  'Copyright' 포함 본문: 0건
  '100자평' 포함 본문: 0건
  'Taboola' 포함 본문: 0건
  '많이 본 뉴스' 포함 본문: 0건
  '업데이트 2025' 포함 본문: 0건

--- 레이블 분포 (불균형 주의) ---
검색 구분 레이블
True     2415
False     197
Name: count, dtype: int64
  비율 12.3 : 1  -> 학습 시 class weight / 리샘플링 고려

--- 섹션 분포 ---
섹션
economy         2393
national         144
politics          36
policy            24
topics            10
distribution       2
it-science         1
opinion            1
Name: count, dtype: int64

--- 상위 기자 ---
작성자
이영관 기자    209
강우량 기자    171
김승현 기자    144
곽창렬 기자    102
조선일보       97
조재희 기자     91
조재현 기자     84
정석우 기자     83
Name: count, dtype: int64


In [7]:
# 육안 검증: 본문 앞/뒤가 깨끗하게 잘렸는지
for i in df.index[:3]:
    r = df.loc[i]
    print(f"[{i}] {r['작성자']} | {r['섹션']} | {r['작성일'].date()} | {r['본문길이']}자")
    print(f"  제목: {r['제목']}")
    print(f"  앞  : {r['본문'][:80]}")
    print(f"  뒤  : ...{r['본문'][-80:]}")
    print()

[0] 이혜진 기자 | national | 2025-01-01 | 914자
  제목: 9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조
  앞  : 무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 ‘푸딩이’가 구조됐다. 동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해
  뒤  : ...이 되지 않은 이들은 3명이다. 푸딩이의 친구였던 정 양과 배씨의 작은딸, 작은딸의 막내아들의 신원이 확인되지 않아 장례를 치르지 못하고 있다.

[1] 조재희 기자 | economy | 2025-01-01 | 665자
  제목: 폴크스바겐, 전기차 정보 부실 관리 논란
  앞  : 전기차 전환에 뒤처지며 창사 87년 만에 처음으로 독일 자국 내 공장 폐쇄에 나선 폴크스바겐이 이번엔 전기차 약 80만대의 운행 데이터와 소유주
  뒤  : ... 취했다. 폴크스바겐 측은 “우리가 아는 한 해커 단체 외에 시스템에 접근한 경우가 없고, 개인 비밀번호나 결제 정보 등은 해당 없다”고 했다.

[2] 김희래 기자 | economy | 2025-01-01 | 5059자
  제목: 최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원
  앞  : 못 받은 양육비, 정부가 선지급… 국가 검진에 C형 간염도 포함 ▲ 최저임금 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 
  뒤  : ... 2000만원으로 확대된다. 이에 따라 세액공제 혜택도 강화돼 기부금 10만원까지 전액 공제, 초과분에 대해서는 16.5% 세액공제가 적용된다.



## 6. 저장

In [8]:
cols = ["제목", "작성자", "작성일", "섹션", "URL", "본문", "본문길이", "검색 구분 레이블"]
out = df[cols].reset_index(drop=True)

out.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {OUT_PATH.resolve()}")
print(f"  {out.shape[0]}행 x {out.shape[1]}열")
print(f"  용량: {OUT_PATH.stat().st_size / 1024 / 1024:.1f} MB (원본 85.5 MB)")
out.head(3)

저장 완료: D:\멋사_부트캠프\클라비_기업프로젝트\data\processed_articles.csv
  2612행 x 8열
  용량: 7.9 MB (원본 85.5 MB)


,제목,작성자,작성일,섹션,URL,본문,본문길이,검색 구분 레이블
0,"9명 가족 잃고, 하염없이 기다리던 푸딩이... 동물단체가 구조",이혜진 기자,2025-01-01,national,https://www.chosun.com/national/national_general/2025/01/01/7S62UVQCONDSPA5G4OQF3NYSDI,무안국제공항 제주항공 여객기 참사로 일가족 9명을 잃은 반려견 ‘푸딩이’가 구조됐다. 동물권보호단체 케어는 지난달 31일 공식 인스타그램을 통해 보호자 없이 마을을 배회하던 푸딩이를 안전하게 보호 중이라고 밝...,914,False
1,"폴크스바겐, 전기차 정보 부실 관리 논란",조재희 기자,2025-01-01,economy,https://www.chosun.com/economy/auto/2025/01/01/S7LNVYUONNHU3KF3WJWFZZ3NVE,전기차 전환에 뒤처지며 창사 87년 만에 처음으로 독일 자국 내 공장 폐쇄에 나선 폴크스바겐이 이번엔 전기차 약 80만대의 운행 데이터와 소유주 정보를 온라인에서 부실 관리했다는 논란에 휩싸였다. 전기차에 이...,665,True
2,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,김희래 기자,2025-01-01,economy,https://www.chosun.com/economy/economy_general/2025/01/01/ADO7O3Q4WJBCJAX6UUEY4MSBV4,"못 받은 양육비, 정부가 선지급… 국가 검진에 C형 간염도 포함 ▲ 최저임금 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7% 인상된다. 주 근로시간 40시간을 기준으로 환산한 월급은...",5059,True
